# Loam heavy worker (Google Colab)

This notebook runs one **heavy** heartbeat of the Loam substrate: a larger budget than the GitHub Actions cron can afford, coordinating with every other worker through the shared store (Turso hub) — stigmergy, no controller.

It is observe-only. It reads public or explicitly authorized sources through the policy layer, never acts on outside systems, and stores no secrets or personal data.

**Setup (once):** in Colab's *Secrets* panel (🔑) add `LOAM_DB_URL`, `LOAM_DB_TOKEN`, `LOAM_PSEUDONYM_SALT` (any random string ≥ 8 chars, the same one used in GitHub Actions), and optionally `GH_PAT`. Enable notebook access for each.


In [ ]:
#@title 1. Node.js 22 (the substrate has zero dependencies)
import subprocess, sys
v = subprocess.run(['bash','-lc','node --version 2>/dev/null || true'], capture_output=True, text=True).stdout.strip()
print('node:', v or 'missing')
major = int(v[1:].split('.')[0]) if v else 0
minor = int(v.split('.')[1]) if v else 0
if major < 22 or (major == 22 and minor < 13):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1 && apt-get install -y nodejs >/dev/null 2>&1
    !node --version


In [ ]:
#@title 2. Clone the repository
REPO = "https://github.com/vector3on/scoutiq.git"  #@param {type:"string"}
BRANCH = "master"  #@param {type:"string"}
!rm -rf loam-src && git clone --depth 1 -b "$BRANCH" "$REPO" loam-src
%cd loam-src/substrate
!node bin/loam.mjs doctor


In [ ]:
#@title 3. Secrets → environment (values never leave this runtime; the substrate stores only their location)
import os
from google.colab import userdata
for k in ['LOAM_DB_URL', 'LOAM_DB_TOKEN', 'LOAM_PSEUDONYM_SALT', 'GH_PAT']:
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception as e:
        print(f'{k}: not set ({type(e).__name__})')
os.environ['LOAM_NODE_NAME'] = 'colab'
print({k: ('set' if os.environ.get(k) else 'unset') for k in ['LOAM_DB_URL','LOAM_DB_TOKEN','LOAM_PSEUDONYM_SALT','GH_PAT']})


In [ ]:
#@title 4. Heavy heartbeat
DOMAIN = "toy"  #@param ["toy", "arxiv-lit", "oss-health"]
BUDGET_SECONDS = 900  #@param {type:"integer"}
!node bin/loam.mjs run --domain "$DOMAIN" --budget "$BUDGET_SECONDS" --role colab


In [ ]:
#@title 5. Report + paste-ready context bundle (for your Claude subscription, not the API)
!node bin/loam.mjs report --domain "$DOMAIN" --last 15
!node bin/loam.mjs bundle --domain "$DOMAIN" --out /content/bundle.md
from IPython.display import Markdown, display
display(Markdown(open('/content/bundle.md').read()))


In [ ]:
#@title 6. Feed a reply back (paste the model's `loam-judgment` block into reply.md first)
REPLY_PATH = "/content/reply.md"  #@param {type:"string"}
import os
if os.path.exists(REPLY_PATH):
    !node bin/loam.mjs ingest-judgment "$REPLY_PATH" --domain "$DOMAIN" --by claude-subscription
else:
    print('no reply file yet')


## 7. External embeddings (v3, optional)

The substrate's built-in embeddings are hashed n-grams (no model). A real sentence encoder sharpens semantic novelty and the learned behavior space. This cell exports the texts that lack a vector, embeds them here with a free open model, and hands the vectors back as `entity.embedded` events (the substrate never loads a model itself). Unmeasured in the toy world; DESIGN.md §9 explains why.


In [ ]:
#@title 7a. Export texts, embed with a free open encoder, import vectors
EMBEDDER = "minilm-l6-v2-384"  #@param {type:"string"}
!node bin/loam.mjs embed-export --domain "$DOMAIN" --out /content/texts.jsonl
import json, os
rows = [json.loads(l) for l in open('/content/texts.jsonl') if l.strip()]
print('texts lacking a vector:', len(rows))
if rows:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        !pip -q install sentence-transformers
        from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    vecs = model.encode([r['text'] for r in rows], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    with open('/content/vectors.jsonl', 'w') as f:
        for r, v in zip(rows, vecs):
            f.write(json.dumps({'entityId': r['entityId'], 'vec': [round(float(x), 5) for x in v]}) + '\n')
    !node bin/loam.mjs embed /content/vectors.jsonl --embedder "$EMBEDDER" --domain "$DOMAIN"
